# SangHyo — MMSE-free Google YDF CNBoost

GitHub의 최신 코드를 clone하고 Training-only EDA, nested-CV 학습, Google Drive 저장을 차례로 실행합니다.

In [ ]:
# Cell 1 - Environment setup
import json
import os
import runpy
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/Pig30nidaE/Google-Ajou-AICapstone.git"
REPO_DIR_NAME = "Google-Ajou-AICapstone"

def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / "SangHyo").is_dir() and (path / "Data").exists():
            return path
    return None

IN_COLAB = in_colab()
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    clone_path = Path("/content") / REPO_DIR_NAME
    os.chdir("/content")
    if clone_path.exists():
        shutil.rmtree(clone_path)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_path)], check=True)
    PROJECT_ROOT = clone_path.resolve()
else:
    PROJECT_ROOT = find_project_root() or Path.cwd().resolve()

os.chdir(PROJECT_ROOT)
DATA_ROOT = Path("/content/drive/MyDrive/GoogleAI_contest/Data") if IN_COLAB else PROJECT_ROOT / "Data"
print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_ROOT    : {DATA_ROOT}")

In [ ]:
# Cell 2 - User inputs
USER_FOLDER = "SangHyo"
EXPERIMENT_FOLDER = "ThreeClass_GoogleYDF_CNBoost"
RUN_MODE = "full"  # "smoke" 또는 "full"
SKIP_TABNET = False   # True이면 Google YDF 네 후보만 실행
SEED = 20260721
RESULTS_ROOT_OVERRIDE = None

In [ ]:
# Cell 3 - Resolve paths
EXPERIMENT_ROOT = (PROJECT_ROOT / USER_FOLDER / EXPERIMENT_FOLDER).resolve()
EDA_PATH = EXPERIMENT_ROOT / "eda.py"
TRAIN_PATH = EXPERIMENT_ROOT / "train.py"
REQUIREMENTS_PATH = EXPERIMENT_ROOT / "requirements_colab.txt"
TRAINING_ROOT = (DATA_ROOT / "1.Training").resolve()
VALIDATION_ROOT = (DATA_ROOT / "2.Validation").resolve()

if RESULTS_ROOT_OVERRIDE is not None:
    RESULTS_ROOT = Path(RESULTS_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    RESULTS_ROOT = Path("/content/drive/MyDrive/SangHyo_CNBoost_Results")
else:
    RESULTS_ROOT = EXPERIMENT_ROOT / "training_outputs"

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
required = [EXPERIMENT_ROOT, EDA_PATH, TRAIN_PATH, REQUIREMENTS_PATH, TRAINING_ROOT, VALIDATION_ROOT]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required path(s):\n" + "\n".join(missing))
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = RESULTS_ROOT / f"{RUN_ID}_{RUN_MODE}_mmse_free_cnboost"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(f"Experiment: {EXPERIMENT_ROOT}")
print(f"Output    : {OUTPUT_DIR}")

In [ ]:
# Cell 4 - Install requirements
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS_PATH)], check=True)
print("Requirements installation complete.")

In [ ]:
# Cell 5 - Training-only EDA and model training
from importlib.metadata import version
import pandas as pd
import torch

if not SKIP_TABNET and not torch.cuda.is_available():
    raise RuntimeError("TabNet을 실행하려면 Colab에서 A100 GPU를 선택해주세요.")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not used'}")
print(f"YDF    : {version('ydf')}")
print(f"TabNet : {version('pytorch-tabnet')}")
print(f"Optuna : {version('optuna')}")

EDA_OUTPUT_DIR = OUTPUT_DIR / "eda"
TRAINING_OUTPUT_DIR = OUTPUT_DIR / "training"
eda_arguments = ["--training-root", str(TRAINING_ROOT), "--output-dir", str(EDA_OUTPUT_DIR), "--seed", str(SEED)]
train_arguments = [
    "--training-root", str(TRAINING_ROOT),
    "--validation-root", str(VALIDATION_ROOT),
    "--output-dir", str(TRAINING_OUTPUT_DIR),
    "--outer-folds", "3",
    "--outer-seeds", "137,1009,2027,4099,8191",
    "--inner-folds", "3",
    "--trials-ydf-multiclass", "36",
    "--trials-ydf-hierarchical", "36",
    "--trials-ydf-random-forest", "24",
    "--trials-ydf-ovr", "24",
    "--trials-tabnet", "20",
    "--seed", str(SEED),
]
if RUN_MODE == "smoke":
    train_arguments.append("--fast")
if SKIP_TABNET:
    train_arguments.append("--skip-tabnet")

def run_python_file(script_path, arguments):
    previous_cwd = Path.cwd()
    previous_argv = sys.argv[:]
    script_dir = str(script_path.parent)
    inserted = script_dir not in sys.path
    if inserted:
        sys.path.insert(0, script_dir)
    os.chdir(script_path.parent)
    sys.argv = [str(script_path), *arguments]
    try:
        return runpy.run_path(str(script_path), run_name="__main__")
    finally:
        sys.argv = previous_argv
        os.chdir(previous_cwd)
        if inserted and script_dir in sys.path:
            sys.path.remove(script_dir)

try:
    print("\n[1/2] Training-only EDA (MMSE/Validation 미접근)")
    run_python_file(EDA_PATH, eda_arguments)
    print("\n[2/2] Google YDF/TabNet nested training")
    run_python_file(TRAIN_PATH, train_arguments)
except BaseException:
    failure_text = traceback.format_exc()
    failure_path = OUTPUT_DIR / "FAILED_TRACEBACK.log"
    failure_path.write_text(failure_text, encoding="utf-8")
    print(failure_text)
    print(f"Error log: {failure_path}")
    raise

report_path = TRAINING_OUTPUT_DIR / "FINAL_REPORT.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
summary = report["primary_nested_repeat_summary"]
rows = []
for metric in ["accuracy", "macro_f1", "roc_auc_ovr_macro", "cn_vs_rest_auc", "balanced_accuracy"]:
    values = summary[metric]
    rows.append({"metric": metric, "mean": values["mean"], "std": values["std"], "min": values["min"], "max": values["max"]})
display(pd.DataFrame(rows))
if report.get("validation_historical") is not None:
    print("Historical validation:", report["validation_historical"]["metrics"])
archive_path = shutil.make_archive(str(RESULTS_ROOT / f"{OUTPUT_DIR.name}_archive"), "zip", root_dir=OUTPUT_DIR)
print(f"Result folder : {OUTPUT_DIR}")
print(f"Drive archive : {archive_path}")